# Day 2: Data Quality & Cleaning Pipeline



"""
🎯 LEARNING OBJECTIVES:
- Implement comprehensive data quality assessment
- Design cleaning procedures for IoT sensor data
- Handle missing values and outliers appropriately
- Create reusable data quality functions

📅 SCHEDULE:
Morning (4 hours):
1. Data Quality Assessment (2 hours)
2. Missing Data Strategy (2 hours)

Afternoon (4 hours):
3. Outlier Detection & Treatment (2 hours)
4. Data Standardization (2 hours)

✅ DELIVERABLES:
- Data quality assessment report
- Comprehensive cleaning pipeline
- Outlier detection and treatment functions
- Standardized datasets ready for analysis
"""


# IMPORTS AND SETUP


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# PySpark imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# Machine learning imports (for outlier detection)
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.stat import Correlation

# Initialize Spark Session (should already exist from Day 1)
try:
    spark.sparkContext.setLogLevel("WARN")
    print("✅ Using existing Spark session")
except:
    spark = (SparkSession.builder
             .appName("SmartCityIoTPipeline-Day2")
             .master("local[*]")
             .config("spark.driver.memory", "4g")
             .config("spark.ui.enabled", "false")
             .config("spark.eventLog.enabled", "false")
             .config("spark.sql.adaptive.enabled", "true")
             .getOrCreate())
    print("✅ Created new Spark session")

print("🔧 Day 2: Data Quality & Cleaning Pipeline")
print("=" * 60)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/13 13:53:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Created new Spark session
🔧 Day 2: Data Quality & Cleaning Pipeline


## SECTION 1: COMPREHENSIVE DATA PROFILING (Morning - 2 hours)


In [2]:
print("\n📊 SECTION 1: COMPREHENSIVE DATA PROFILING")
print("=" * 60)

# Load cleaned data from Day 1 (or reload if needed)
data_dir = "../data/raw"



📊 SECTION 1: COMPREHENSIVE DATA PROFILING


## TODO 1.1: Load all datasets with error handling


In [3]:
def load_all_datasets():
    """Load all sensor datasets with consistent error handling"""
    datasets = {}

    try:
        # Load each dataset
        datasets['zones'] = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{data_dir}/city_zones.csv")
        datasets['traffic'] = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{data_dir}/traffic_sensors.csv")
        datasets['air_quality'] = spark.read.json(f"{data_dir}/air_quality.json")
        datasets['weather'] = spark.read.parquet(f"{data_dir}/weather_data.parquet")
        datasets['energy'] = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{data_dir}/energy_meters.csv")
        datasets['occupancy'] = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{data_dir}/occupancy_data.csv")
        datasets['fiscal'] = spark.read.option("header", "true").option("inferSchema", "true").csv(f"{data_dir}/fiscal_data.csv")

        # Convert timestamp columns to proper format
        for name, df in datasets.items():
            if name != 'zones' and 'timestamp' in df.columns:
                datasets[name] = df.withColumn("timestamp", F.to_timestamp(F.col("timestamp")))

        print("✅ All datasets loaded successfully")
        return datasets

    except Exception as e:
        print(f"❌ Error loading datasets: {str(e)}")
        return {}


In [4]:
datasets = load_all_datasets()


✅ All datasets loaded successfully


## TODO 1.2: Advanced Data Quality Metrics (45 minutes)


In [26]:
"""
🎯 TASK: Create comprehensive data quality profiling functions
💡 HINT: Look beyond basic missing values - consider temporal patterns, distributions
📚 CONCEPTS: Data profiling, quality metrics, statistical validation
"""

def comprehensive_data_profile(df, dataset_name, time_col="timestamp"):
    """
    Generate comprehensive data quality profile

    Args:
        df: Spark DataFrame to profile
        dataset_name: Name for reporting
        time_col: Timestamp column name

    Returns:
        Dictionary with quality metrics
    """
    print(f"\n🔍 Comprehensive Profile: {dataset_name}")
    print("-" * 50)

    # Basic statistics
    total_rows = df.count()
    total_cols = len(df.columns)

    profile = {
        'dataset_name': dataset_name,
        'total_rows': total_rows,
        'total_columns': total_cols,
        'memory_usage_mb': 0,  # Estimate
        'quality_issues': []
    }

    # TODO: Calculate missing value patterns
    print("📋 Missing Value Analysis:")
    missing_analysis = {}
    for col in df.columns:
        missing_count = df.filter(F.col(col).isNull()).count()
        missing_pct = (missing_count / total_rows) * 100 if total_rows > 0 else 0
        missing_analysis[col] = {'count': missing_count, 'percentage': missing_pct}

        if missing_pct > 5:  # Flag columns with >5% missing
            profile['quality_issues'].append(f"High missing values in {col}: {missing_pct:.2f}%")

        if missing_count > 0:
            print(f"   {col}: {missing_count:,} ({missing_pct:.2f}%)")

    profile['missing_analysis'] = missing_analysis

    # TODO: Temporal data gaps (if timestamp column exists)
    if time_col in df.columns:
        print("⏰ Temporal Analysis:")

        # Get time range
        time_stats = df.agg(
            F.min(time_col).alias('min_time'),
            F.max(time_col).alias('max_time'),
            F.count(time_col).alias('time_count')
        ).collect()[0]

        print(f"   Time Range: {time_stats['min_time']} to {time_stats['max_time']}")
        print(f"   Records with timestamps: {time_stats['time_count']:,}")

        # TODO: Check for temporal gaps
        # Calculate expected vs actual record counts
        if dataset_name == 'traffic':
            expected_interval_minutes = 5
        elif dataset_name == 'air_quality':
            expected_interval_minutes = 15
        elif dataset_name == 'weather':
            expected_interval_minutes = 30
        elif dataset_name == 'energy':
            expected_interval_minutes = 10
        else:
            expected_interval_minutes = 15

        # TODO: Add your gap detection logic here
        print(f"   Expected interval: {expected_interval_minutes} minutes")
        # HINT: Calculate expected number of records based on time range and interval

    # TODO: Numeric column distributions
    numeric_cols = [field.name for field in df.schema.fields
                   if field.dataType in [IntegerType(), DoubleType(), FloatType(), LongType()]]

    if numeric_cols:
        print("📈 Numeric Column Analysis:")
        # Get basic statistics for numeric columns
        stats_df = df.select(numeric_cols).describe()
        stats_df.show()

        # TODO: Check for suspicious patterns in numeric data
        for col in numeric_cols:
            if col not in ['location_lat', 'location_lon']:
                # Check for columns with very low variance (potentially stuck sensors)
                variance_check = df.agg(F.variance(col).alias('variance')).collect()[0]['variance']
                if variance_check is not None and variance_check < 0.001:
                    profile['quality_issues'].append(f"Very low variance in {col}: {variance_check}")

    # TODO: Categorical column analysis
    categorical_cols = [field.name for field in df.schema.fields
                       if field.dataType == StringType() and field.name not in [time_col]]

    if categorical_cols:
        print("📂 Categorical Column Analysis:")
        for col in categorical_cols:
            distinct_count = df.select(col).distinct().count()
            print(f"   {col}: {distinct_count} distinct values")

            # Show top values
            if distinct_count < 20:
                top_values = df.groupBy(col).count().orderBy(F.desc("count")).limit(5)
                print(f"      Top values:")
                top_values.show(5, truncate=False)

    # TODO: Check for duplicate records
    duplicate_count = total_rows - df.dropDuplicates().count()
    if duplicate_count > 0:
        profile['quality_issues'].append(f"Duplicate records found: {duplicate_count}")
        print(f"🔄 Duplicate Records: {duplicate_count:,}")

    # TODO: Data freshness (for time series data)
    if time_col in df.columns:
        latest_record = df.agg(F.max(time_col).alias('latest')).collect()[0]['latest']
        if latest_record:
            hours_old = (datetime.now() - latest_record).total_seconds() / 3600
            print(f"📅 Data Freshness: Latest record is {hours_old:.1f} hours old")

    return profile

# TODO: Profile all datasets


In [27]:
print("🔍 Starting comprehensive data profiling...")

profiles = {}


🔍 Starting comprehensive data profiling...


In [28]:
for name, df in datasets.items():
    if df is not None:
        try:
            profiles[name] = comprehensive_data_profile(df, name)
        except Exception as e:
            print(f"❌ Error profiling {name}: {str(e)}")



🔍 Comprehensive Profile: zones
--------------------------------------------------
📋 Missing Value Analysis:
📈 Numeric Column Analysis:


26/09/13 14:58:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-------------------+--------------------+--------------------+--------------------+------------------+
|summary|            lat_min|             lat_max|             lon_min|             lon_max|        population|
+-------+-------------------+--------------------+--------------------+--------------------+------------------+
|  count|              36000|               36000|               36000|               36000|             36000|
|   mean|  40.75935789444529|  40.760200000001056|  -73.96555921051029|  -73.96466447367668| 3022.003583333333|
| stddev|0.04606103919488944|0.046061039701610976|0.049078050635065436|0.049078050635950575|2855.8303511974086|
|    min|              40.68|           40.680842|              -74.05|          -74.049105|                 0|
|    max|          40.839158|               40.84|          -73.880895|              -73.88|             12000|
+-------+-------------------+--------------------+--------------------+--------------------+------------

## TODO 1.3: Sensor Health Analysis (45 minutes)


In [29]:
"""
🎯 TASK: Identify sensors with potential operational issues
💡 HINT: Look for patterns that indicate sensor malfunctions
📚 CONCEPTS: Sensor diagnostics, operational monitoring, health scoring
"""

def analyze_sensor_health(df, sensor_id_col, value_cols, time_col="timestamp"):
    """
    Analyze individual sensor health and identify problematic sensors.

    Args:
        df: DataFrame with sensor data
        sensor_id_col: Column name for sensor ID
        value_cols: List of measurement columns to analyze
        time_col: Timestamp column

    Returns:
        DataFrame with sensor health metrics
    """

    print("\n🏥 Sensor Health Analysis")
    print("-" * 30)

    # TODO: Calculate health metrics per sensor
    health_metrics = df.groupBy(sensor_id_col).agg(
        F.count("*").alias("total_readings"),
        F.min(time_col).alias("first_reading"),
        F.max(time_col).alias("last_reading")
    )

    # Add missing data percentage per sensor
    for col in value_cols:
        if col in df.columns:
            missing_col_name = f"{col}_missing_pct"

            missing_counts = df.groupBy(sensor_id_col).agg(
                F.sum(
                    F.when(F.col(col).isNull(), 1).otherwise(0)
                ).alias(f"{col}_missing")
            )

            health_metrics = health_metrics.join(
                missing_counts,
                sensor_id_col,
                "left"
            )

            health_metrics = health_metrics.withColumn(
                missing_col_name,
                F.round(
                    (F.col(f"{col}_missing") /
                     F.col("total_readings")) * 100,
                    2
                )
            )

    # TODO: Add variance analysis (detect stuck sensors)
    for col in value_cols:
        if col in df.columns:
            variance_col_name = f"{col}_variance"

            sensor_variance = df.groupBy(sensor_id_col).agg(
                F.variance(col).alias(variance_col_name)
            )

            health_metrics = health_metrics.join(
                sensor_variance,
                sensor_id_col,
                "left"
            )

    # TODO: Calculate data gaps (irregular reporting)
    gap_window = (
        Window
        .partitionBy(sensor_id_col)
        .orderBy(time_col)
    )

    gap_df = (
        df
        .withColumn(
            "previous_timestamp",
            F.lag(time_col).over(gap_window)
        )
        .withColumn(
            "gap_minutes",
            (
                F.unix_timestamp(F.col(time_col)) -
                F.unix_timestamp(F.col("previous_timestamp"))
            ) / 60
        )
    )

    gap_metrics = gap_df.groupBy(sensor_id_col).agg(
        F.round(F.avg("gap_minutes"), 2).alias("avg_gap_minutes"),
        F.round(F.max("gap_minutes"), 2).alias("max_gap_minutes")
    )

    health_metrics = health_metrics.join(
        gap_metrics,
        sensor_id_col,
        "left"
    )

    # TODO: Create overall health score

    # Average missing-data percentage
    missing_pct_cols = [
        f"{col}_missing_pct"
        for col in value_cols
        if f"{col}_missing_pct" in health_metrics.columns
    ]

    if missing_pct_cols:
        total_missing_pct = F.lit(0.0)

        for col_name in missing_pct_cols:
            total_missing_pct = (
                total_missing_pct +
                F.coalesce(F.col(col_name), F.lit(0.0))
            )

        avg_missing_pct = (
            total_missing_pct / len(missing_pct_cols)
        )
    else:
        avg_missing_pct = F.lit(0.0)

    # Detect sensors that appear stuck
    variance_cols = [
        f"{col}_variance"
        for col in value_cols
        if f"{col}_variance" in health_metrics.columns
    ]

    stuck_condition = F.lit(False)

    for col_name in variance_cols:
        stuck_condition = (
            stuck_condition |
            (F.coalesce(F.col(col_name), F.lit(0.0)) == 0)
        )

    # Reporting gaps: penalize sensors whose largest gap
    # is much larger than their normal average gap
    irregular_reporting = (
        F.col("max_gap_minutes") >
        (F.col("avg_gap_minutes") * 3)
    )

    health_metrics = health_metrics.withColumn(
        "health_score",
        F.round(
            F.greatest(
                F.lit(0.0),
                F.lit(100.0)
                - avg_missing_pct
                - F.when(stuck_condition, 25).otherwise(0)
                - F.when(irregular_reporting, 15).otherwise(0)
            ),
            2
        )
    )

    # TODO: Flag problematic sensors
    health_metrics = health_metrics.withColumn(
        "status",
        F.when(F.col("health_score") > 80, "healthy")
         .when(F.col("health_score") > 60, "warning")
         .otherwise("critical")
    )

    return health_metrics

    # TODO: Analyze health for each sensor type

    sensor_health_results = {}

    if "traffic" in datasets:
        sensor_health_results["traffic"] = analyze_sensor_health(
            datasets["traffic"],
            "sensor_id",
            ["vehicle_count", "avg_speed"]
        )

    if "air_quality" in datasets:
        sensor_health_results["air_quality"] = analyze_sensor_health(
            datasets["air_quality"],
            "sensor_id",
            ["pm25", "pm10", "no2", "co", "temperature", "humidity"]
        )

    if "energy" in datasets:
        sensor_health_results["energy"] = analyze_sensor_health(
            datasets["energy"],
            "meter_id",
            ["power_consumption", "voltage", "current", "power_factor"]
        )

    if "occupancy" in datasets:
        sensor_health_results["occupancy"] = analyze_sensor_health(
            datasets["occupancy"],
            "sensor_id",
            ["available_rooms", "occupied_rooms", "guests"]
        )

    if "fiscal" in datasets:
        sensor_health_results["fiscal"] = analyze_sensor_health(
            datasets["fiscal"],
            "sensor_id",
            ["expense", "revenue"]
        )

    for sensor_type, health_df in sensor_health_results.items():
        print(f"\n📊 {sensor_type.upper()} SENSOR HEALTH")
        health_df.groupBy("status").count().show()


In [30]:


print("🏥 Analyzing sensor health across all datasets...")

sensor_health_results = {}

# Traffic sensors


🏥 Analyzing sensor health across all datasets...


In [31]:
if 'traffic' in datasets:
    traffic_health = analyze_sensor_health(
        datasets['traffic'],
        'sensor_id',
        ['vehicle_count', 'avg_speed']
    )
    sensor_health_results['traffic'] = traffic_health

    print("🚗 Traffic Sensor Health Summary:")
    traffic_health.groupBy("status").count().show()

# TODO: Analyze other sensor types

# Air quality sensors
if 'air_quality' in datasets:
    air_quality_health = analyze_sensor_health(
        datasets['air_quality'],
        "sensor_id",
        ["pm25", "pm10", "no2", "co", "temperature", "humidity"]
    )

    print("🌫️ Air Quality Sensor Health Summary:")
    air_quality_health.groupBy("status").count().show()


# Weather sensors
if 'weather' in datasets:
    weather_health = analyze_sensor_health(
        datasets['weather'],
        "station_id",
        [
            "temperature",
            "humidity",
            "wind_speed",
            "wind_direction",
            "precipitation",
            "pressure"
        ]
    )

    print("🌦️ Weather Sensor Health Summary:")
    weather_health.groupBy("status").count().show()


# Energy sensors
if 'energy' in datasets:
    energy_health = analyze_sensor_health(
        datasets['energy'],
        "meter_id",
        [
            "power_consumption",
            "voltage",
            "current",
            "power_factor"
        ]
    )

    print("⚡ Energy Sensor Health Summary:")
    energy_health.groupBy("status").count().show()


# Occupancy sensors
if 'occupancy' in datasets:
    occupancy_health = analyze_sensor_health(
        datasets['occupancy'],
        "sensor_id",
        [
            "available_rooms",
            "occupied_rooms",
            "guests"
        ]
    )

    print("🏨 Occupancy Sensor Health Summary:")
    occupancy_health.groupBy("status").count().show()


# Fiscal sensors
if 'fiscal' in datasets:
    fiscal_health = analyze_sensor_health(
        datasets['fiscal'],
        "sensor_id",
        [
            "expense",
            "revenue"
        ]
    )

    print("💰 Fiscal Sensor Health Summary:")
    fiscal_health.groupBy("status").count().show()


🏥 Sensor Health Analysis
------------------------------
🚗 Traffic Sensor Health Summary:
+-------+-----+
| status|count|
+-------+-----+
|healthy| 3000|
+-------+-----+


🏥 Sensor Health Analysis
------------------------------
🌫️ Air Quality Sensor Health Summary:


+-------+-----+
| status|count|
+-------+-----+
|healthy| 1800|
+-------+-----+


🏥 Sensor Health Analysis
------------------------------
🌦️ Weather Sensor Health Summary:
+-------+-----+
| status|count|
+-------+-----+
|healthy| 1200|
+-------+-----+


🏥 Sensor Health Analysis
------------------------------
⚡ Energy Sensor Health Summary:
+-------+-----+
| status|count|
+-------+-----+
|healthy| 2400|
+-------+-----+


🏥 Sensor Health Analysis
------------------------------
🏨 Occupancy Sensor Health Summary:
+-------+-----+
| status|count|
+-------+-----+
|healthy| 1200|
+-------+-----+


🏥 Sensor Health Analysis
------------------------------
💰 Fiscal Sensor Health Summary:
+-------+-----+
| status|count|
+-------+-----+
|healthy| 1200|
+-------+-----+



## SECTION 2: MISSING DATA STRATEGY (Morning - 2 hours)


In [16]:
print("\n" + "=" * 60)
print("🕳️ SECTION 2: MISSING DATA HANDLING STRATEGY")
print("=" * 60)



🕳️ SECTION 2: MISSING DATA HANDLING STRATEGY


26/09/13 04:47:44 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 658401 ms exceeds timeout 120000 ms
26/09/13 04:47:44 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/13 04:47:52 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora

## TODO 2.1: Missing Data Pattern Analysis (60 minutes)


In [5]:
def analyze_missing_patterns(df, time_col="timestamp", sensor_col=None):
    """
    Analyze patterns in missing data.

    Args:
        df: DataFrame to analyze
        time_col: Timestamp column
        sensor_col: Sensor ID column (if applicable)

    Returns:
        Dictionary with missing data insights
    """

    print("\n🔍 Missing Data Pattern Analysis")
    print("-" * 40)

    patterns = {
        "temporal": {},
        "sensor": None
    }

    # TODO: Temporal patterns in missing data
    if time_col in df.columns:
        print("⏰ Temporal Missing Data Patterns:")

        df_with_time_features = (
            df
            .withColumn("hour", F.hour(time_col))
            .withColumn("day_of_week", F.dayofweek(time_col))
            .withColumn("date", F.to_date(time_col))
        )

        numeric_cols = [
            field.name
            for field in df.schema.fields
            if isinstance(
                field.dataType,
                (IntegerType, DoubleType, FloatType)
            )
        ]

        # TODO: Check missing data by hour of day
        for col in numeric_cols[:3]:
            missing_by_hour = (
                df_with_time_features
                .groupBy("hour")
                .agg(
                    F.count("*").alias("total_records"),
                    F.sum(
                        F.when(F.col(col).isNull(), 1).otherwise(0)
                    ).alias("missing_count")
                )
                .withColumn(
                    "missing_pct",
                    F.round(
                        (
                            F.col("missing_count") /
                            F.col("total_records")
                        ) * 100,
                        2
                    )
                )
                .orderBy("hour")
            )

            print(f"\n   Missing data by hour for {col}:")
            missing_by_hour.show(24)

            # Save the result
            patterns["temporal"][col] = missing_by_hour

            # TODO: Identify problematic hours
            high_missing_hours = missing_by_hour.filter(
                F.col("missing_pct") > 10
            )

            if high_missing_hours.count() > 0:
                print(f"   ⚠️ High missing data hours for {col}:")
                high_missing_hours.show()

    # TODO: Sensor-specific missing patterns
    if sensor_col and sensor_col in df.columns:
        print("\n📡 Sensor-specific Missing Patterns:")

        sensor_missing = df.groupBy(sensor_col).agg(
            F.count("*").alias("total_readings")
        )

        numeric_cols = [
            field.name
            for field in df.schema.fields
            if isinstance(
                field.dataType,
                (IntegerType, DoubleType, FloatType)
            )
        ]

        for col in numeric_cols[:2]:
            col_missing = df.groupBy(sensor_col).agg(
                F.sum(
                    F.when(F.col(col).isNull(), 1).otherwise(0)
                ).alias(f"{col}_missing")
            )

            sensor_missing = sensor_missing.join(
                col_missing,
                sensor_col,
                "left"
            )

        # Calculate percentages
        for col in numeric_cols[:2]:
            sensor_missing = sensor_missing.withColumn(
                f"{col}_missing_pct",
                F.round(
                    (
                        F.col(f"{col}_missing") /
                        F.col("total_readings")
                    ) * 100,
                    2
                )
            )

        print("   Sensors with highest missing data:")

        if numeric_cols:
            sensor_missing.orderBy(
                F.desc(f"{numeric_cols[0]}_missing_pct")
            ).show(10)

        # Save the sensor-level analysis
        patterns["sensor"] = sensor_missing

    return patterns

# TODO: Analyze missing patterns for each dataset

missing_pattern_results = {}

if "traffic" in datasets:
    print("\n🚗 TRAFFIC")
    missing_pattern_results["traffic"] = analyze_missing_patterns(
        datasets["traffic"],
        sensor_col="sensor_id"
    )

if "air_quality" in datasets:
    print("\n🌫️ AIR QUALITY")
    missing_pattern_results["air_quality"] = analyze_missing_patterns(
        datasets["air_quality"],
        sensor_col="sensor_id"
    )

if "weather" in datasets:
    print("\n🌦️ WEATHER")
    missing_pattern_results["weather"] = analyze_missing_patterns(
        datasets["weather"],
        sensor_col="station_id"
    )

if "energy" in datasets:
    print("\n⚡ ENERGY")
    missing_pattern_results["energy"] = analyze_missing_patterns(
        datasets["energy"],
        sensor_col="meter_id"
    )

if "occupancy" in datasets:
    print("\n🏨 OCCUPANCY")
    missing_pattern_results["occupancy"] = analyze_missing_patterns(
        datasets["occupancy"],
        sensor_col="sensor_id"
    )

if "fiscal" in datasets:
    print("\n💰 FISCAL")
    missing_pattern_results["fiscal"] = analyze_missing_patterns(
        datasets["fiscal"],
        sensor_col="sensor_id"
    )


🚗 TRAFFIC

🔍 Missing Data Pattern Analysis
----------------------------------------
⏰ Temporal Missing Data Patterns:

   Missing data by hour for location_lat:
+----+-------------+-------------+-----------+
|hour|total_records|missing_count|missing_pct|
+----+-------------+-------------+-----------+
|   0|         1500|            0|        0.0|
|   1|         1500|            0|        0.0|
|   2|         1488|            0|        0.0|
|   3|         1512|            0|        0.0|
|   4|         1500|            0|        0.0|
|   5|         1500|            0|        0.0|
|   6|         1500|            0|        0.0|
|   7|         1500|            0|        0.0|
|   8|         1500|            0|        0.0|
|   9|         1500|            0|        0.0|
|  10|         1500|            0|        0.0|
|  11|         1500|            0|        0.0|
|  12|         1500|            0|        0.0|
|  13|         1500|            0|        0.0|
|  14|         1500|            0|     

In [7]:
missing_pattern_results = {}

for name, df in datasets.items():
    if df is not None and name != "zones":
        try:
            sensor_col = None

            if "sensor_id" in df.columns:
                sensor_col = "sensor_id"
            elif "station_id" in df.columns:
                sensor_col = "station_id"
            elif "meter_id" in df.columns:
                sensor_col = "meter_id"

            missing_pattern_results[name] = analyze_missing_patterns(
                df,
                sensor_col=sensor_col
            )

        except Exception as e:
            print(
                f"❌ Error analyzing missing patterns for {name}: {str(e)}"
            )


🔍 Missing Data Pattern Analysis
----------------------------------------
⏰ Temporal Missing Data Patterns:

   Missing data by hour for location_lat:
+----+-------------+-------------+-----------+
|hour|total_records|missing_count|missing_pct|
+----+-------------+-------------+-----------+
|   0|         1500|            0|        0.0|
|   1|         1500|            0|        0.0|
|   2|         1488|            0|        0.0|
|   3|         1512|            0|        0.0|
|   4|         1500|            0|        0.0|
|   5|         1500|            0|        0.0|
|   6|         1500|            0|        0.0|
|   7|         1500|            0|        0.0|
|   8|         1500|            0|        0.0|
|   9|         1500|            0|        0.0|
|  10|         1500|            0|        0.0|
|  11|         1500|            0|        0.0|
|  12|         1500|            0|        0.0|
|  13|         1500|            0|        0.0|
|  14|         1500|            0|        0.0|
|  

## TODO 2.2: Time Series Interpolation (60 minutes)


In [9]:
def interpolate_time_series_gaps(
    df,
    value_columns,
    time_col="timestamp",
    sensor_col=None,
    max_gap_hours=6
):
    """
    Implement time series interpolation for missing values.

    Args:
        df: DataFrame with time series data
        value_columns: List of columns to interpolate
        time_col: Timestamp column
        sensor_col: Sensor ID column for per-sensor interpolation
        max_gap_hours: Maximum gap size to interpolate

    Returns:
        DataFrame with interpolated values
    """

    print("\n🔧 Time Series Interpolation")
    print("-" * 30)

    result_df = df

    # Create appropriate window
    if sensor_col and sensor_col in df.columns:
        window_spec = (
            Window
            .partitionBy(sensor_col)
            .orderBy(time_col)
        )
    else:
        window_spec = Window.orderBy(time_col)

    for col in value_columns:
        if col not in df.columns:
            continue

        print(f"   Interpolating {col}...")

        # Previous non-null value
        previous_value = F.last(
            F.col(col),
            ignorenulls=True
        ).over(
            window_spec.rowsBetween(
                Window.unboundedPreceding,
                -1
            )
        )

        # Timestamp of previous non-null value
        previous_time = F.last(
            F.when(
                F.col(col).isNotNull(),
                F.col(time_col)
            ),
            ignorenulls=True
        ).over(
            window_spec.rowsBetween(
                Window.unboundedPreceding,
                -1
            )
        )

        # Calculate gap from previous valid reading
        gap_hours = (
            F.unix_timestamp(F.col(time_col)) -
            F.unix_timestamp(previous_time)
        ) / 3600

        # Fill only when:
        # 1. original value is missing
        # 2. a previous value exists
        # 3. gap is within maximum allowed size
        result_df = result_df.withColumn(
            f"{col}_filled",
            F.when(
                F.col(col).isNull()
                & previous_value.isNotNull()
                & (gap_hours <= max_gap_hours),
                previous_value
            ).otherwise(F.col(col))
        )

        # Track whether interpolation occurred
        result_df = result_df.withColumn(
            f"{col}_interpolated",
            F.when(
                F.col(col).isNull()
                & F.col(f"{col}_filled").isNotNull(),
                True
            ).otherwise(False)
        )

    # Report interpolation statistics
    print("📊 Interpolation Summary:")

    for col in value_columns:
        filled_col = f"{col}_filled"
        interpolated_col = f"{col}_interpolated"

        if (
            filled_col in result_df.columns
            and interpolated_col in result_df.columns
        ):
            interpolated_count = (
                result_df
                .filter(F.col(interpolated_col) == True)
                .count()
            )

            total_missing = (
                df
                .filter(F.col(col).isNull())
                .count()
            )

            print(
                f"   {col}: "
                f"{interpolated_count}/{total_missing} "
                f"missing values interpolated"
            )

    return result_df

In [10]:
if 'traffic' in datasets:
    print("🚗 Testing interpolation on traffic data...")
    traffic_interpolated = interpolate_time_series_gaps(
        datasets['traffic'],
        ['vehicle_count', 'avg_speed'],
        sensor_col='sensor_id'
    )

    # Show before/after comparison
    print("\n📊 Before/After Interpolation Comparison:")
    comparison = datasets['traffic'].agg(
        F.sum(F.when(F.col("vehicle_count").isNull(), 1).otherwise(0)).alias("vehicle_count_missing_before")
    ).join(
        traffic_interpolated.agg(
            F.sum(F.when(F.col("vehicle_count_filled").isNull(), 1).otherwise(0)).alias("vehicle_count_missing_after")
        )
    )
    comparison.show()


🚗 Testing interpolation on traffic data...

🔧 Time Series Interpolation
------------------------------
   Interpolating vehicle_count...
   Interpolating avg_speed...
📊 Interpolation Summary:
   vehicle_count: 0/0 missing values interpolated
   avg_speed: 0/0 missing values interpolated

📊 Before/After Interpolation Comparison:
+----------------------------+---------------------------+
|vehicle_count_missing_before|vehicle_count_missing_after|
+----------------------------+---------------------------+
|                           0|                          0|
+----------------------------+---------------------------+



## SECTION 3: OUTLIER DETECTION & TREATMENT (Afternoon - 2 hours)


In [11]:
print("\n" + "=" * 60)
print("🎯 SECTION 3: OUTLIER DETECTION & TREATMENT")
print("=" * 60)



🎯 SECTION 3: OUTLIER DETECTION & TREATMENT


## TODO 3.1: Statistical Outlier Detection (60 minutes)


In [12]:
"""
🎯 TASK: Implement multiple outlier detection methods
💡 HINT: Different methods work better for different data distributions
📚 CONCEPTS: IQR method, Z-score, isolation forest, domain-specific rules
"""

def detect_statistical_outliers(
    df,
    columns,
    methods=["iqr", "zscore"],
    sensor_col=None
):
    """
    Detect outliers using statistical methods.

    Args:
        df: DataFrame to analyze
        columns: List of numeric columns to check
        methods: Detection methods to use
        sensor_col: Sensor ID column for per-sensor analysis

    Returns:
        DataFrame with outlier flags added
    """

    print("\n🔍 Statistical Outlier Detection")
    print("-" * 35)

    result_df = df

    for col in columns:
        if col not in df.columns:
            continue

        print(f"   Analyzing {col}...")

        # =========================================================
        # IQR METHOD
        # =========================================================
        if "iqr" in methods:

            if sensor_col and sensor_col in df.columns:
                # Calculate IQR separately for each sensor
                sensor_iqr_stats = (
                    df.groupBy(sensor_col)
                    .agg(
                        F.expr(
                            f"percentile_approx({col}, 0.25)"
                        ).alias("_q1"),
                        F.expr(
                            f"percentile_approx({col}, 0.75)"
                        ).alias("_q3")
                    )
                    .withColumn(
                        "_iqr",
                        F.col("_q3") - F.col("_q1")
                    )
                    .withColumn(
                        "_lower_bound",
                        F.col("_q1") - (1.5 * F.col("_iqr"))
                    )
                    .withColumn(
                        "_upper_bound",
                        F.col("_q3") + (1.5 * F.col("_iqr"))
                    )
                )

                result_df = result_df.join(
                    sensor_iqr_stats,
                    sensor_col,
                    "left"
                )

                result_df = result_df.withColumn(
                    f"{col}_outlier_iqr",
                    F.when(
                        (F.col(col) < F.col("_lower_bound"))
                        | (F.col(col) > F.col("_upper_bound")),
                        True
                    ).otherwise(False)
                )

                result_df = result_df.drop(
                    "_q1",
                    "_q3",
                    "_iqr",
                    "_lower_bound",
                    "_upper_bound"
                )

            else:
                # Global IQR
                quartiles = (
                    df.select(
                        F.expr(
                            f"percentile_approx({col}, 0.25)"
                        ).alias("q1"),
                        F.expr(
                            f"percentile_approx({col}, 0.75)"
                        ).alias("q3")
                    )
                    .collect()[0]
                )

                q1 = quartiles["q1"]
                q3 = quartiles["q3"]

                if q1 is not None and q3 is not None:
                    iqr = q3 - q1
                    lower_bound = q1 - (1.5 * iqr)
                    upper_bound = q3 + (1.5 * iqr)

                    result_df = result_df.withColumn(
                        f"{col}_outlier_iqr",
                        F.when(
                            (F.col(col) < lower_bound)
                            | (F.col(col) > upper_bound),
                            True
                        ).otherwise(False)
                    )

            if f"{col}_outlier_iqr" in result_df.columns:
                outlier_count = (
                    result_df
                    .filter(F.col(f"{col}_outlier_iqr") == True)
                    .count()
                )

                print(
                    f"      IQR method: "
                    f"{outlier_count} outliers detected"
                )

        # =========================================================
        # Z-SCORE METHOD
        # =========================================================
        if "zscore" in methods:

            if sensor_col and sensor_col in df.columns:
                sensor_stats = (
                    df.groupBy(sensor_col)
                    .agg(
                        F.mean(col).alias("_mean"),
                        F.stddev(col).alias("_stddev")
                    )
                )

                result_df = result_df.join(
                    sensor_stats,
                    sensor_col,
                    "left"
                )

                result_df = result_df.withColumn(
                    f"{col}_zscore",
                    F.when(
                        F.col("_stddev") > 0,
                        F.abs(
                            (F.col(col) - F.col("_mean"))
                            / F.col("_stddev")
                        )
                    )
                ).withColumn(
                    f"{col}_outlier_zscore",
                    F.when(
                        F.col(f"{col}_zscore") > 3,
                        True
                    ).otherwise(False)
                )

                result_df = result_df.drop(
                    "_mean",
                    "_stddev"
                )

            else:
                stats = (
                    df.select(
                        F.mean(col).alias("mean_val"),
                        F.stddev(col).alias("stddev_val")
                    )
                    .collect()[0]
                )

                mean_val = stats["mean_val"]
                stddev_val = stats["stddev_val"]

                if (
                    mean_val is not None
                    and stddev_val is not None
                    and stddev_val > 0
                ):
                    result_df = result_df.withColumn(
                        f"{col}_zscore",
                        F.abs(
                            (F.col(col) - mean_val) / stddev_val
                        )
                    ).withColumn(
                        f"{col}_outlier_zscore",
                        F.when(
                            F.col(f"{col}_zscore") > 3,
                            True
                        ).otherwise(False)
                    )

            if f"{col}_outlier_zscore" in result_df.columns:
                outlier_count = (
                    result_df
                    .filter(
                        F.col(f"{col}_outlier_zscore") == True
                    )
                    .count()
                )

                print(
                    f"      Z-Score method: "
                    f"{outlier_count} outliers detected"
                )

    return result_df

# TODO: Domain-specific outlier rules
def detect_domain_outliers(df, dataset_type):
    """
    Apply domain-specific outlier detection rules.

    Args:
        df: DataFrame to analyze
        dataset_type: Type of sensor data

    Returns:
        DataFrame with domain-specific outlier flags
    """

    print(
        f"\n🏭 Domain-Specific Outlier Detection: "
        f"{dataset_type}"
    )
    print("-" * 45)

    result_df = df

    # =============================================================
    # TRAFFIC
    # =============================================================
    if dataset_type == "traffic":

        if "avg_speed" in df.columns:
            result_df = result_df.withColumn(
                "speed_outlier_domain",
                F.when(
                    (F.col("avg_speed") < 0)
                    | (F.col("avg_speed") > 120),
                    True
                ).otherwise(False)
            )

        if "vehicle_count" in df.columns:
            result_df = result_df.withColumn(
                "vehicle_count_outlier_domain",
                F.when(
                    F.col("vehicle_count") < 0,
                    True
                ).otherwise(False)
            )

    # =============================================================
    # AIR QUALITY
    # =============================================================
    elif dataset_type == "air_quality":

        if "pm25" in df.columns:
            result_df = result_df.withColumn(
                "pm25_outlier_domain",
                F.when(
                    (F.col("pm25") < 0)
                    | (F.col("pm25") > 500),
                    True
                ).otherwise(False)
            )

        if "pm10" in df.columns:
            result_df = result_df.withColumn(
                "pm10_outlier_domain",
                F.when(
                    F.col("pm10") < 0,
                    True
                ).otherwise(False)
            )

        if "no2" in df.columns:
            result_df = result_df.withColumn(
                "no2_outlier_domain",
                F.when(
                    F.col("no2") < 0,
                    True
                ).otherwise(False)
            )

        if "co" in df.columns:
            result_df = result_df.withColumn(
                "co_outlier_domain",
                F.when(
                    F.col("co") < 0,
                    True
                ).otherwise(False)
            )

        if "humidity" in df.columns:
            result_df = result_df.withColumn(
                "humidity_outlier_domain",
                F.when(
                    (F.col("humidity") < 0)
                    | (F.col("humidity") > 100),
                    True
                ).otherwise(False)
            )

    # =============================================================
    # WEATHER
    # =============================================================
    elif dataset_type == "weather":

        if "humidity" in df.columns:
            result_df = result_df.withColumn(
                "humidity_outlier_domain",
                F.when(
                    (F.col("humidity") < 0)
                    | (F.col("humidity") > 100),
                    True
                ).otherwise(False)
            )

        if "wind_speed" in df.columns:
            result_df = result_df.withColumn(
                "wind_speed_outlier_domain",
                F.when(
                    F.col("wind_speed") < 0,
                    True
                ).otherwise(False)
            )

        if "precipitation" in df.columns:
            result_df = result_df.withColumn(
                "precipitation_outlier_domain",
                F.when(
                    F.col("precipitation") < 0,
                    True
                ).otherwise(False)
            )

    # =============================================================
    # ENERGY
    # =============================================================
    elif dataset_type == "energy":

        if "power_consumption" in df.columns:
            result_df = result_df.withColumn(
                "power_outlier_domain",
                F.when(
                    F.col("power_consumption") < 0,
                    True
                ).otherwise(False)
            )

    # =============================================================
    # OCCUPANCY
    # =============================================================
    elif dataset_type == "occupancy":

        # Fixed the original duplicate occupied_rooms check
        if (
            "occupied_rooms" in df.columns
            and "available_rooms" in df.columns
        ):
            result_df = result_df.withColumn(
                "occupancy_outlier_domain",
                F.when(
                    (F.col("available_rooms") < 0)
                    | (F.col("occupied_rooms") < 0),
                    True
                ).otherwise(False)
            )

        if "guests" in df.columns:
            result_df = result_df.withColumn(
                "guests_outlier_domain",
                F.when(
                    F.col("guests") < 0,
                    True
                ).otherwise(False)
            )

    # =============================================================
    # FISCAL
    # =============================================================
    elif dataset_type == "fiscal":

        if "revenue" in df.columns:
            result_df = result_df.withColumn(
                "revenue_outlier_domain",
                F.when(
                    F.col("revenue") < 0,
                    True
                ).otherwise(False)
            )

        if "expense" in df.columns:
            result_df = result_df.withColumn(
                "expense_outlier_domain",
                F.when(
                    F.col("expense") < 0,
                    True
                ).otherwise(False)
            )

    # =============================================================
    # SUMMARY
    # =============================================================
    outlier_cols = [
        col
        for col in result_df.columns
        if col.endswith("_outlier_domain")
    ]

    if outlier_cols:
        for col in outlier_cols:
            outlier_count = (
                result_df
                .filter(F.col(col) == True)
                .count()
            )

            print(
                f"   {col}: "
                f"{outlier_count} outliers detected"
            )

    return result_df

# TODO: Test outlier detection on all datasets


In [13]:
print("🎯 Testing outlier detection across all datasets...")

outlier_results = {}

for name, df in datasets.items():

    if df is not None and name != "zones":

        try:
            # Determine sensor identifier
            sensor_col = None

            if "sensor_id" in df.columns:
                sensor_col = "sensor_id"

            elif "station_id" in df.columns:
                sensor_col = "station_id"

            elif "meter_id" in df.columns:
                sensor_col = "meter_id"

            # Identify numeric measurement columns
            numeric_cols = [
                field.name
                for field in df.schema.fields
                if isinstance(
                    field.dataType,
                    (IntegerType, DoubleType, FloatType)
                )
                and field.name not in [
                    "location_lat",
                    "location_lon"
                ]
            ]

            if numeric_cols:

                # Statistical detection
                df_with_outliers = (
                    detect_statistical_outliers(
                        df,
                        numeric_cols[:3],
                        sensor_col=sensor_col
                    )
                )

                # Domain-specific detection
                df_with_all_outliers = (
                    detect_domain_outliers(
                        df_with_outliers,
                        name
                    )
                )

                outlier_results[name] = (
                    df_with_all_outliers
                )

                print(
                    f"\n✅ Outlier detection "
                    f"completed for {name}"
                )

        except Exception as e:

            print(
                f"❌ Error in outlier detection "
                f"for {name}: {str(e)}"
            )


🎯 Testing outlier detection across all datasets...

🔍 Statistical Outlier Detection
-----------------------------------
   Analyzing vehicle_count...
      IQR method: 964 outliers detected
      Z-Score method: 2 outliers detected
   Analyzing avg_speed...
      IQR method: 2200 outliers detected
      Z-Score method: 14 outliers detected

🏭 Domain-Specific Outlier Detection: traffic
---------------------------------------------
   speed_outlier_domain: 0 outliers detected
   vehicle_count_outlier_domain: 0 outliers detected

✅ Outlier detection completed for traffic

🔍 Statistical Outlier Detection
-----------------------------------
   Analyzing co...
      IQR method: 77 outliers detected
      Z-Score method: 0 outliers detected
   Analyzing humidity...
      IQR method: 0 outliers detected
      Z-Score method: 0 outliers detected
   Analyzing no2...
      IQR method: 71 outliers detected
      Z-Score method: 1 outliers detected

🏭 Domain-Specific Outlier Detection: air_quality


## TODO 3.2: Outlier Treatment Strategies (60 minutes)


In [14]:
"""
🎯 TASK: Implement different strategies for handling detected outliers
💡 HINT: Consider business impact when choosing treatment methods
📚 CONCEPTS: Capping, removal, imputation, flagging
"""

def treat_outliers(
    df,
    treatment_strategy="cap",
    outlier_columns=None
):
    """
    Apply outlier treatment strategies.

    Args:
        df: DataFrame with outlier flags
        treatment_strategy: 'cap', 'remove', 'impute', or 'flag_only'
        outlier_columns: List of outlier flag columns to process

    Returns:
        DataFrame with outliers treated
    """

    print(
        f"\n🔧 Outlier Treatment Strategy: "
        f"{treatment_strategy}"
    )
    print("-" * 45)

    valid_strategies = {
        "cap",
        "remove",
        "impute",
        "flag_only"
    }

    if treatment_strategy not in valid_strategies:
        raise ValueError(
            f"Invalid treatment strategy: {treatment_strategy}"
        )

    result_df = df

    if outlier_columns is None:
        outlier_columns = [
            col
            for col in df.columns
            if "_outlier_" in col
        ]

    # Map special domain flag names to their real data columns
    domain_column_map = {
        "speed_outlier_domain": "avg_speed",
        "vehicle_count_outlier_domain": "vehicle_count",
        "pm25_outlier_domain": "pm25",
        "pm10_outlier_domain": "pm10",
        "no2_outlier_domain": "no2",
        "co_outlier_domain": "co",
        "humidity_outlier_domain": "humidity",
        "wind_speed_outlier_domain": "wind_speed",
        "precipitation_outlier_domain": "precipitation",
        "power_outlier_domain": "power_consumption",
        "guests_outlier_domain": "guests",
        "revenue_outlier_domain": "revenue",
        "expense_outlier_domain": "expense",
    }

    treatment_stats = {}

    for outlier_col in outlier_columns:

        # Determine the measurement column associated
        # with the outlier flag
        if outlier_col in domain_column_map:
            original_col = domain_column_map[outlier_col]
        else:
            original_col = outlier_col.split(
                "_outlier_"
            )[0]

        if original_col not in df.columns:
            print(
                f"   ⚠️ Skipping {outlier_col}: "
                f"no matching measurement column"
            )
            continue

        outlier_count = (
            df
            .filter(F.col(outlier_col) == True)
            .count()
        )

        treatment_stats[outlier_col] = {
            "original_outliers": outlier_count
        }

        # No need to calculate anything when there
        # are no outliers for this flag
        if outlier_count == 0:
            treatment_stats[outlier_col]["action"] = (
                "none_found"
            )
            continue

        # =========================================================
        # REMOVE
        # =========================================================
        if treatment_strategy == "remove":

            result_df = result_df.filter(
                ~F.coalesce(
                    F.col(outlier_col),
                    F.lit(False)
                )
            )

            treatment_stats[outlier_col]["action"] = (
                "removed"
            )

        # =========================================================
        # CAP
        # =========================================================
        elif treatment_strategy == "cap":

            percentiles = (
                df.select(
                    F.expr(
                        f"percentile_approx("
                        f"{original_col}, 0.05)"
                    ).alias("p5"),
                    F.expr(
                        f"percentile_approx("
                        f"{original_col}, 0.95)"
                    ).alias("p95")
                )
                .collect()[0]
            )

            p5 = percentiles["p5"]
            p95 = percentiles["p95"]

            if p5 is not None and p95 is not None:

                capped_col = f"{original_col}_capped"

                # If another outlier rule already created
                # the capped column, use that as the base.
                source_col = (
                    capped_col
                    if capped_col in result_df.columns
                    else original_col
                )

                result_df = result_df.withColumn(
                    capped_col,
                    F.when(
                        F.col(outlier_col) == True,
                        F.when(
                            F.col(source_col) < p5,
                            F.lit(p5)
                        )
                        .when(
                            F.col(source_col) > p95,
                            F.lit(p95)
                        )
                        .otherwise(
                            F.col(source_col)
                        )
                    ).otherwise(
                        F.col(source_col)
                    )
                )

                treatment_stats[outlier_col][
                    "action"
                ] = "capped"

        # =========================================================
        # IMPUTE
        # =========================================================
        elif treatment_strategy == "impute":

            median_val = (
                df
                .filter(
                    F.col(outlier_col) == False
                )
                .select(
                    F.expr(
                        f"percentile_approx("
                        f"{original_col}, 0.5)"
                    ).alias("median")
                )
                .collect()[0]["median"]
            )

            if median_val is not None:

                imputed_col = (
                    f"{original_col}_imputed"
                )

                source_col = (
                    imputed_col
                    if imputed_col in result_df.columns
                    else original_col
                )

                result_df = result_df.withColumn(
                    imputed_col,
                    F.when(
                        F.col(outlier_col) == True,
                        F.lit(median_val)
                    ).otherwise(
                        F.col(source_col)
                    )
                )

                treatment_stats[outlier_col][
                    "action"
                ] = "imputed"

        # =========================================================
        # FLAG ONLY
        # =========================================================
        elif treatment_strategy == "flag_only":

            treatment_stats[outlier_col][
                "action"
            ] = "flagged_only"

    # =============================================================
    # SUMMARY
    # =============================================================
    print("📊 Treatment Summary:")

    for col, stats in treatment_stats.items():

        action = stats.get("action", "none")
        count = stats.get(
            "original_outliers",
            0
        )

        print(
            f"   {col}: "
            f"{count} outliers {action}"
        )

    return result_df


if "traffic" in outlier_results:

    print(
        "\n🚗 Testing outlier treatment "
        "on traffic data..."
    )

    traffic_treated = treat_outliers(
        outlier_results["traffic"],
        treatment_strategy="cap"
    )

    print(
        "✅ Outlier treatment completed "
        "for traffic data"
    )


🚗 Testing outlier treatment on traffic data...

🔧 Outlier Treatment Strategy: cap
---------------------------------------------
📊 Treatment Summary:
   vehicle_count_outlier_iqr: 964 outliers capped
   vehicle_count_outlier_zscore: 2 outliers capped
   avg_speed_outlier_iqr: 2200 outliers capped
   avg_speed_outlier_zscore: 14 outliers capped
   speed_outlier_domain: 0 outliers none_found
   vehicle_count_outlier_domain: 0 outliers none_found
✅ Outlier treatment completed for traffic data


## SECTION 4: DATA STANDARDIZATION (Afternoon - 2 hours)


In [15]:
print("\n" + "=" * 60)
print("📏 SECTION 4: DATA STANDARDIZATION")
print("=" * 60)



📏 SECTION 4: DATA STANDARDIZATION


## TODO 4.1: Unit Standardization (60 minutes)


In [16]:
"""
🎯 TASK: Ensure consistent units across all measurements
💡 HINT: Different sensors might use different units for similar measurements
📚 CONCEPTS: Unit conversion, standardization, metadata management
"""

def standardize_measurement_units(df, sensor_type):
    """
    Standardize measurement units for a sensor type.

    Args:
        df: DataFrame with sensor measurements
        sensor_type: Type of sensor
                     (traffic, air_quality, weather,
                      energy, occupancy, fiscal)

    Returns:
        DataFrame with standardized units
    """

    print(f"\n📏 Standardizing units for {sensor_type} sensors")
    print("-" * 40)

    result_df = df
    conversions_applied = []

    # =========================================================
    # TRAFFIC
    # =========================================================
    if sensor_type == "traffic":

        if "avg_speed" in df.columns:
            result_df = (
                result_df
                .withColumn(
                    "avg_speed_kmh",
                    F.col("avg_speed")
                )
                .withColumn(
                    "avg_speed_unit",
                    F.lit("km/h")
                )
            )

            conversions_applied.append(
                "Speed: km/h"
            )

    # =========================================================
    # AIR QUALITY
    # =========================================================
    elif sensor_type == "air_quality":

        if "pm25" in df.columns:
            result_df = (
                result_df
                .withColumn(
                    "pm25_ugm3",
                    F.col("pm25")
                )
                .withColumn(
                    "pm25_unit",
                    F.lit("μg/m³")
                )
            )

            conversions_applied.append(
                "PM2.5: μg/m³"
            )

        if "pm10" in df.columns:
            result_df = (
                result_df
                .withColumn(
                    "pm10_ugm3",
                    F.col("pm10")
                )
                .withColumn(
                    "pm10_unit",
                    F.lit("μg/m³")
                )
            )

            conversions_applied.append(
                "PM10: μg/m³"
            )

        if "temperature" in df.columns:
            result_df = (
                result_df
                .withColumn(
                    "temperature_celsius",
                    F.col("temperature")
                )
                .withColumn(
                    "temperature_unit",
                    F.lit("°C")
                )
            )

            conversions_applied.append(
                "Temperature: °C"
            )

    # =========================================================
    # WEATHER
    # =========================================================
    elif sensor_type == "weather":

        if "temperature" in df.columns:
            result_df = (
                result_df
                .withColumn(
                    "temperature_celsius",
                    F.col("temperature")
                )
                .withColumn(
                    "temperature_unit",
                    F.lit("°C")
                )
            )

            conversions_applied.append(
                "Temperature: °C"
            )

        if "wind_speed" in df.columns:
            result_df = (
                result_df
                .withColumn(
                    "wind_speed_kmh",
                    F.col("wind_speed")
                )
                .withColumn(
                    "wind_speed_unit",
                    F.lit("km/h")
                )
            )

            conversions_applied.append(
                "Wind Speed: km/h"
            )

        if "pressure" in df.columns:
            result_df = (
                result_df
                .withColumn(
                    "pressure_hpa",
                    F.col("pressure")
                )
                .withColumn(
                    "pressure_unit",
                    F.lit("hPa")
                )
            )

            conversions_applied.append(
                "Pressure: hPa"
            )

    # =========================================================
    # ENERGY
    # =========================================================
    elif sensor_type == "energy":

        if "power_consumption" in df.columns:
            result_df = (
                result_df
                .withColumn(
                    "power_consumption_kw",
                    F.col("power_consumption")
                )
                .withColumn(
                    "power_unit",
                    F.lit("kW")
                )
            )

            conversions_applied.append(
                "Power: kW"
            )

        if "voltage" in df.columns:
            result_df = (
                result_df
                .withColumn(
                    "voltage_v",
                    F.col("voltage")
                )
                .withColumn(
                    "voltage_unit",
                    F.lit("V")
                )
            )

            conversions_applied.append(
                "Voltage: V"
            )

        if "current" in df.columns:
            result_df = (
                result_df
                .withColumn(
                    "current_a",
                    F.col("current")
                )
                .withColumn(
                    "current_unit",
                    F.lit("A")
                )
            )

            conversions_applied.append(
                "Current: A"
            )

    # =========================================================
    # OCCUPANCY
    # =========================================================
    elif sensor_type == "occupancy":

        if "available_rooms" in df.columns:
            result_df = result_df.withColumn(
                "available_rooms_unit",
                F.lit("rooms")
            )

            conversions_applied.append(
                "Available Rooms: count"
            )

        if "occupied_rooms" in df.columns:
            result_df = result_df.withColumn(
                "occupied_rooms_unit",
                F.lit("rooms")
            )

            conversions_applied.append(
                "Occupied Rooms: count"
            )

        if "guests" in df.columns:
            result_df = result_df.withColumn(
                "guests_unit",
                F.lit("people")
            )

            conversions_applied.append(
                "Guests: people"
            )

    # =========================================================
    # FISCAL
    # =========================================================
    elif sensor_type == "fiscal":

        if "revenue" in df.columns:
            result_df = result_df.withColumn(
                "revenue_unit",
                F.lit("currency")
            )

            conversions_applied.append(
                "Revenue: currency"
            )

        if "expense" in df.columns:
            result_df = result_df.withColumn(
                "expense_unit",
                F.lit("currency")
            )

            conversions_applied.append(
                "Expense: currency"
            )

    # =========================================================
    # SUMMARY
    # =========================================================
    print("🔄 Unit conversions applied:")

    if conversions_applied:
        for conversion in conversions_applied:
            print(f"   ✅ {conversion}")
    else:
        print(
            "   ℹ️ No unit-specific conversions required"
        )

    return result_df




In [17]:
# TODO: Apply unit standardization to all datasets

standardized_datasets = {}

for name, df in datasets.items():
    if df is not None and name != 'zones':
        try:
            standardized_df = standardize_measurement_units(df, name)
            standardized_datasets[name] = standardized_df
            print(f"✅ Unit standardization completed for {name}")
        except Exception as e:
            print(f"❌ Error standardizing {name}: {str(e)}")



📏 Standardizing units for traffic sensors
----------------------------------------
🔄 Unit conversions applied:
   ✅ Speed: km/h
✅ Unit standardization completed for traffic

📏 Standardizing units for air_quality sensors
----------------------------------------
🔄 Unit conversions applied:
   ✅ PM2.5: μg/m³
   ✅ PM10: μg/m³
   ✅ Temperature: °C
✅ Unit standardization completed for air_quality

📏 Standardizing units for weather sensors
----------------------------------------
🔄 Unit conversions applied:
   ✅ Temperature: °C
   ✅ Wind Speed: km/h
   ✅ Pressure: hPa
✅ Unit standardization completed for weather

📏 Standardizing units for energy sensors
----------------------------------------
🔄 Unit conversions applied:
   ✅ Power: kW
   ✅ Voltage: V
   ✅ Current: A
✅ Unit standardization completed for energy

📏 Standardizing units for occupancy sensors
----------------------------------------
🔄 Unit conversions applied:
   ✅ Available Rooms: count
   ✅ Occupied Rooms: count
   ✅ Guests: pe

## TODO 4.2: Data Lineage Tracking (60 minutes)


In [18]:
import builtins
import json


def add_data_lineage(
    df,
    transformations_applied,
    data_quality_score=None
):
    """
    Add data lineage and quality tracking columns.

    Args:
        df: DataFrame to augment
        transformations_applied: List of transformation descriptions
        data_quality_score: Overall quality score (0-1)

    Returns:
        DataFrame with lineage metadata
    """

    print("\n📋 Adding Data Lineage Tracking")
    print("-" * 35)

    result_df = df

    # =========================================================
    # PROCESSING TIMESTAMP
    # =========================================================
    result_df = result_df.withColumn(
        "processed_at",
        F.current_timestamp()
    )

    # =========================================================
    # TRANSFORMATION HISTORY
    # =========================================================
    transformations_json = json.dumps(
        transformations_applied
    )

    result_df = result_df.withColumn(
        "transformations_applied",
        F.lit(transformations_json)
    )

    # =========================================================
    # DATA QUALITY SCORE
    # =========================================================
    if data_quality_score is not None:
        result_df = result_df.withColumn(
            "data_quality_score",
            F.lit(float(data_quality_score))
        )

    # =========================================================
    # ROW COMPLETENESS SCORE
    # =========================================================
    numeric_cols = [
        field.name
        for field in df.schema.fields
        if isinstance(
            field.dataType,
            (IntegerType, DoubleType, FloatType)
        )
    ]

    if numeric_cols:

        non_null_expressions = [
            F.when(
                F.col(col).isNotNull(),
                1
            ).otherwise(0)
            for col in numeric_cols
        ]

        non_null_count = builtins.sum(
            non_null_expressions
        )

        total_numeric_cols = len(numeric_cols)

        result_df = result_df.withColumn(
            "row_completeness_score",
            non_null_count / F.lit(total_numeric_cols)
        )

    else:
        result_df = result_df.withColumn(
            "row_completeness_score",
            F.lit(1.0)
        )

    # =========================================================
    # RECORD-LEVEL OUTLIER FLAG
    # =========================================================
    outlier_flag_cols = [
        col
        for col in result_df.columns
        if "_outlier_" in col
    ]

    if outlier_flag_cols:

        outlier_condition = F.lit(False)

        for col in outlier_flag_cols:
            outlier_condition = (
                outlier_condition
                | F.coalesce(
                    F.col(col),
                    F.lit(False)
                )
            )

        result_df = result_df.withColumn(
            "has_outliers",
            outlier_condition
        )

    else:
        result_df = result_df.withColumn(
            "has_outliers",
            F.lit(False)
        )

    # =========================================================
    # RECORD-LEVEL INTERPOLATION FLAG
    # =========================================================
    interpolated_flag_cols = [
        col
        for col in result_df.columns
        if col.endswith("_interpolated")
    ]

    if interpolated_flag_cols:

        interpolated_condition = F.lit(False)

        for col in interpolated_flag_cols:
            interpolated_condition = (
                interpolated_condition
                | F.coalesce(
                    F.col(col),
                    F.lit(False)
                )
            )

        result_df = result_df.withColumn(
            "has_interpolated_values",
            interpolated_condition
        )

    else:
        result_df = result_df.withColumn(
            "has_interpolated_values",
            F.lit(False)
        )

    print("📊 Lineage metadata added:")
    print("   ✅ Processing timestamp")
    print("   ✅ Transformation history")
    print("   ✅ Data quality score")
    print("   ✅ Row completeness score")
    print("   ✅ Record-level quality flags")

    return result_df

In [19]:
# TODO: Add lineage to all processed datasets

final_datasets = {}

for name, df in standardized_datasets.items():

    try:
        transformations = [
            "missing_data_analysis",
            "outlier_detection",
            "unit_standardization"
        ]

        total_rows = df.count()

        if total_rows > 0:

            # Find numeric measurement columns
            numeric_cols = [
                field.name
                for field in df.schema.fields
                if isinstance(
                    field.dataType,
                    (
                        IntegerType,
                        DoubleType,
                        FloatType
                    )
                )
            ]

            if numeric_cols:

                # Count all numeric values
                total_possible_values = (
                    total_rows * len(numeric_cols)
                )

                missing_expressions = [
                    F.sum(
                        F.when(
                            F.col(col).isNull(),
                            1
                        ).otherwise(0)
                    )
                    for col in numeric_cols
                ]

                missing_row = df.select(
                    *[
                        expr.alias(f"missing_{i}")
                        for i, expr
                        in enumerate(
                            missing_expressions
                        )
                    ]
                ).collect()[0]

                total_missing = builtins.sum(
                    value or 0
                    for value in missing_row
                )

                quality_score = (
                    1 -
                    (
                        total_missing
                        / total_possible_values
                    )
                )

            else:
                quality_score = 1.0

        else:
            quality_score = 0.0

        final_df = add_data_lineage(
            df,
            transformations,
            quality_score
        )

        final_datasets[name] = final_df

        print(
            f"✅ Lineage tracking added to {name} "
            f"(quality score: "
            f"{quality_score:.2f})"
        )

    except Exception as e:

        print(
            f"❌ Error adding lineage to "
            f"{name}: {str(e)}"
        )


📋 Adding Data Lineage Tracking
-----------------------------------
📊 Lineage metadata added:
   ✅ Processing timestamp
   ✅ Transformation history
   ✅ Data quality score
   ✅ Row completeness score
   ✅ Record-level quality flags
✅ Lineage tracking added to traffic (quality score: 1.00)

📋 Adding Data Lineage Tracking
-----------------------------------
📊 Lineage metadata added:
   ✅ Processing timestamp
   ✅ Transformation history
   ✅ Data quality score
   ✅ Row completeness score
   ✅ Record-level quality flags
✅ Lineage tracking added to air_quality (quality score: 1.00)

📋 Adding Data Lineage Tracking
-----------------------------------
📊 Lineage metadata added:
   ✅ Processing timestamp
   ✅ Transformation history
   ✅ Data quality score
   ✅ Row completeness score
   ✅ Record-level quality flags
✅ Lineage tracking added to weather (quality score: 1.00)

📋 Adding Data Lineage Tracking
-----------------------------------
📊 Lineage metadata added:
   ✅ Processing timestamp
   ✅ T

## DAY 2 DELIVERABLES & VALIDATION


In [35]:
# =============================================================================
# DAY 2 DELIVERABLES & VALIDATION
# =============================================================================

print("\n" + "=" * 60)
print("📋 DAY 2 COMPLETION CHECKLIST")
print("=" * 60)


def validate_day2_completion():
    """Validate that Day 2 objectives have been met."""

    import builtins

    checklist = {
        "data_profiling_completed": False,
        "sensor_health_analyzed": False,
        "missing_data_patterns_identified": False,
        "interpolation_implemented": False,
        "outlier_detection_working": False,
        "outlier_treatment_applied": False,
        "units_standardized": False,
        "data_lineage_tracked": False,
        "quality_scores_calculated": False
    }

    try:

       # =========================================================
       # # 1. DATA PROFILING
       # # =========================================================
       if (
            "profiles" in globals()
            and len(profiles) > 0
            and "comprehensive_data_profile" in globals()
            and callable(comprehensive_data_profile)
        ):
            checklist["data_profiling_completed"] = True


        # =========================================================
        # 2. SENSOR HEALTH
        # =========================================================

            sensor_health_objects = [
                "traffic_health",
                "air_quality_health",
                "weather_health",
                "energy_health",
                "occupancy_health",
                "fiscal_health"
            ]

            health_results_found = [
                name
                for name in sensor_health_objects
                if name in globals()
            ]

            if (
                "analyze_sensor_health" in globals()
                and callable(analyze_sensor_health)
                and len(health_results_found) > 0
            ):
                checklist["sensor_health_analyzed"] = True


        # =========================================================
        # 3. MISSING DATA PATTERNS
        # =========================================================

            if (
                "analyze_missing_patterns" in globals()
                and callable(analyze_missing_patterns)
            ):
                checklist[
                    "missing_data_patterns_identified"
                ] = True


        # =========================================================
        # 4. INTERPOLATION
        # =========================================================

            if (
                "interpolate_time_series_gaps" in globals()
                and callable(interpolate_time_series_gaps)
            ):
                checklist["interpolation_implemented"] = True


        # =========================================================
        # 5. OUTLIER DETECTION
        # =========================================================

            if (
                "outlier_results" in globals()
                and len(outlier_results) > 0
            ):
                checklist["outlier_detection_working"] = True


        # =========================================================
        # 6. OUTLIER TREATMENT
        # =========================================================

            if "traffic_treated" in globals():

                treated_columns = [
                    col
                    for col in traffic_treated.columns
                    if (
                        col.endswith("_capped")
                        or col.endswith("_imputed")
                    )
                ]

                if treated_columns:
                    checklist["outlier_treatment_applied"] = True


        # =========================================================
        # 7. UNIT STANDARDIZATION
        # =========================================================

            if (
                "standardized_datasets" in globals()
                and len(standardized_datasets) > 0
            ):

                standardized_columns_found = False

                for df in standardized_datasets.values():

                    if any(
                        col.endswith(
                            (
                                "_unit",
                                "_kmh",
                                "_ugm3",
                                "_celsius",
                                "_hpa",
                                "_kw",
                                "_v",
                                "_a"
                            )
                        )
                        for col in df.columns
                    ):
                        standardized_columns_found = True
                        break

                checklist[
                    "units_standardized"
                ] = standardized_columns_found


        # =========================================================
        # 8. DATA LINEAGE
        # =========================================================

            if (
                "final_datasets" in globals()
                and len(final_datasets) > 0
            ):

                lineage_complete = all(
                    "processed_at" in df.columns
                    and "transformations_applied" in df.columns
                    and "row_completeness_score" in df.columns
                    for df in final_datasets.values()
                )

                checklist[
                    "data_lineage_tracked"
                ] = lineage_complete


        # =========================================================
        # 9. QUALITY SCORES
        # =========================================================

            if (
                "final_datasets" in globals()
                and len(final_datasets) > 0
            ):

                quality_scores_present = all(
                    "data_quality_score" in df.columns
                    for df in final_datasets.values()
                )

                checklist[
                    "quality_scores_calculated"
                ] = quality_scores_present


    except Exception as e:
        print(f"❌ Validation error: {str(e)}")


    # =============================================================
    # DISPLAY RESULTS
    # =============================================================

    print("✅ COMPLETION STATUS:")

    for item, status in checklist.items():

        status_icon = "✅" if status else "❌"

        print(
            f"   {status_icon} "
            f"{item.replace('_', ' ').title()}"
        )


    # Use Python's built-in sum instead of PySpark sum
    completion_rate = (
        builtins.sum(checklist.values())
        / len(checklist)
        * 100
    )

    print(
        f"\n📊 Overall Completion: "
        f"{completion_rate:.1f}%"
    )


    if completion_rate >= 70:

        print(
            "🎉 Great progress! "
            "You're ready for Day 3!"
        )

        print("\n📈 KEY INSIGHTS FROM DAY 2:")

        print(
            "- Data quality patterns identified "
            "across sensors"
        )

        print(
            "- Missing data handling strategies "
            "implemented"
        )

        print(
            "- Outlier detection and treatment "
            "procedures established"
        )

        print(
            "- Standardized data formats for "
            "consistent analysis"
        )

        print(
            "- Data lineage and quality metadata "
            "added"
        )

    else:

        print(
            "📝 Please review incomplete items "
            "before proceeding to Day 3."
        )


    return checklist


# =============================================================================
# RUN DAY 2 VALIDATION
# =============================================================================

completion_status = validate_day2_completion()


📋 DAY 2 COMPLETION CHECKLIST
✅ COMPLETION STATUS:
   ✅ Data Profiling Completed
   ✅ Sensor Health Analyzed
   ✅ Missing Data Patterns Identified
   ✅ Interpolation Implemented
   ✅ Outlier Detection Working
   ✅ Outlier Treatment Applied
   ✅ Units Standardized
   ✅ Data Lineage Tracked
   ✅ Quality Scores Calculated

📊 Overall Completion: 100.0%
🎉 Great progress! You're ready for Day 3!

📈 KEY INSIGHTS FROM DAY 2:
- Data quality patterns identified across sensors
- Missing data handling strategies implemented
- Outlier detection and treatment procedures established
- Standardized data formats for consistent analysis
- Data lineage and quality metadata added


In [32]:
print("Possible profiling variables:")
for name in globals():
    if "profil" in name.lower():
        print("  ", name)

print("\nPossible sensor health variables:")
for name in globals():
    if "health" in name.lower():
        print("  ", name)

Possible profiling variables:
   comprehensive_data_profile
   profiles

Possible sensor health variables:
   analyze_sensor_health
   sensor_health_results
   traffic_health
   air_quality_health
   weather_health
   energy_health
   occupancy_health
   fiscal_health


In [36]:
# Run the validation

completion_status = validate_day2_completion()


✅ COMPLETION STATUS:
   ✅ Data Profiling Completed
   ✅ Sensor Health Analyzed
   ✅ Missing Data Patterns Identified
   ✅ Interpolation Implemented
   ✅ Outlier Detection Working
   ✅ Outlier Treatment Applied
   ✅ Units Standardized
   ✅ Data Lineage Tracked
   ✅ Quality Scores Calculated

📊 Overall Completion: 100.0%
🎉 Great progress! You're ready for Day 3!

📈 KEY INSIGHTS FROM DAY 2:
- Data quality patterns identified across sensors
- Missing data handling strategies implemented
- Outlier detection and treatment procedures established
- Standardized data formats for consistent analysis
- Data lineage and quality metadata added


## SAVE CLEANED DATA FOR DAY 3


In [37]:
# =============================================================================
# SAVE CLEANED DATA FOR DAY 3
# =============================================================================

print("\n💾 SAVING CLEANED DATA FOR DAY 3")
print("=" * 40)

import os

# =============================================================================
# TODO: Save the cleaned datasets for use in Day 3
# =============================================================================

cleaned_data_dir = "../data/processed"

# Create the processed-data directory if it does not exist
os.makedirs(cleaned_data_dir, exist_ok=True)

for name, df in final_datasets.items():

    try:

        # Save each cleaned dataset as Parquet
        output_path = (
            f"{cleaned_data_dir}/"
            f"{name}_cleaned.parquet"
        )

        df.write.mode("overwrite").parquet(
            output_path
        )

        print(
            f"✅ Saved cleaned {name} data "
            f"to {output_path}"
        )

    except Exception as e:

        print(
            f"❌ Error saving {name}: "
            f"{str(e)}"
        )


💾 SAVING CLEANED DATA FOR DAY 3


✅ Saved cleaned traffic data to ../data/processed/traffic_cleaned.parquet


✅ Saved cleaned air_quality data to ../data/processed/air_quality_cleaned.parquet
✅ Saved cleaned weather data to ../data/processed/weather_cleaned.parquet


✅ Saved cleaned energy data to ../data/processed/energy_cleaned.parquet
✅ Saved cleaned occupancy data to ../data/processed/occupancy_cleaned.parquet
✅ Saved cleaned fiscal data to ../data/processed/fiscal_cleaned.parquet


## Day 2 Findings, Cleaning Rules & Sensor Notes

### Key Findings
- All six time-series datasets were evaluated for missing values and data-quality issues.
- No missing measurement values were found in the generated datasets.
- Statistical outliers were identified using both IQR and Z-score methods.
- Domain-based validation found no invalid traffic speeds, negative vehicle counts, or other major domain violations.
- Statistical outliers were treated as unusual observations rather than automatically being classified as bad data.
- Weather data contains timestamps extending into the future relative to the analysis date. This should be reviewed as a data-quality issue.

### Cleaning and Quality Rules
- Missing-data patterns were analyzed across each time-series dataset.
- Time-series interpolation logic was implemented for missing values within an acceptable time gap.
- IQR and Z-score methods were used to identify statistical outliers.
- Domain-specific rules were used to identify values outside reasonable ranges.
- Measurement units were standardized across traffic, air quality, weather, energy, occupancy, and fiscal datasets.
- Data-lineage and quality-score metadata were added to support traceability.

### Sensor Health
- All sensors evaluated by the sensor-health analysis were classified as healthy.
- No sensors were classified as warning or critical.
- No sensors currently appear to require maintenance based on the Day 2 health analysis.

### Items for Further Review
- Day 1 geographic zone mapping produced 36,079 mapped traffic records from 36,000 source records, suggesting that some geographic zone boundaries overlap.
- Future-dated weather records should be reviewed to determine whether they are intentional generated test data or should be treated as a quality issue.

## NEXT STEPS


In [ ]:
print("\n" + "=" * 60)
print("🚀 WHAT'S NEXT?")
print("=" * 60)

print("""
📅 DAY 3 PREVIEW: Time Series Analysis & Feature Engineering

Tomorrow you'll work on:
1. 📈 Seasonal decomposition and temporal pattern analysis
2. 🔗 Cross-sensor correlation studies
3. ⚙️ Advanced feature engineering for machine learning
4. 📊 Trend detection and forecasting preparation
5. 🎯 Predictive feature creation

📚 RECOMMENDED PREPARATION:
- Review time series analysis concepts
- Understand correlation and causation
- Read about feature engineering best practices
- Familiarize yourself with window functions in Spark

💡 KEY TAKEAWAYS FROM DAY 2:
- Data quality assessment is crucial for IoT data
- Missing data patterns often reveal operational insights
- Outlier detection requires both statistical and domain knowledge
- Standardization enables consistent cross-sensor analysis
- Data lineage tracking is essential for production systems

🤝 QUESTIONS FOR REFLECTION:
- Which sensors showed the most quality issues and why?
- What patterns did you notice in missing data?
- How might seasonal factors affect your cleaning strategies?
- What domain knowledge would improve outlier detection?

💾 SAVE YOUR WORK:
- Commit your notebook with findings and insights
- Document any custom cleaning rules you developed
- Note any sensors that may need maintenance
""")

print("\n💾 Don't forget to save your notebook and commit your changes!")
